# DBSCAN Preparation + Clustering

This notebook prepares `5_test_candidate_cleaned_final.csv` for DBSCAN and then runs DBSCAN.

## Workflow
1. Load dataset
2. Remove identifier columns
3. Engineer datetime features
4. Impute missing values
5. Encode categorical features + scale numeric features
6. Use k-distance diagnostics to pick `eps`
7. Fit DBSCAN and review cluster/noise profile


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA

from sklearn.metrics import silhouette_score


In [ ]:
DATA_PATH = "../data/processed/5_test_candidate_cleaned_final.csv"

df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
df.head(3)


In [ ]:
# 1) Drop identifier columns that should not drive clustering distance
id_cols = [
    "CANDIDATE_ATTEMPT_ID",
    "CANDIDATE_ID",
    "RESULT_ID",
    "TEST_CENTER_ID",
]

existing_id_cols = [c for c in id_cols if c in df.columns]
df_model = df.drop(columns=existing_id_cols).copy()

print("Dropped ID columns:", existing_id_cols)
print("Remaining shape:", df_model.shape)


In [ ]:
# 2) Engineer datetime columns into numeric features, then remove raw text fields
for col in ["EXAM_START", "ACCOUNT_SETUP_COMPLETE_DATE", "CREDENTIAL_DATE"]:
    if col in df_model.columns:
        dt = pd.to_datetime(df_model[col], errors="coerce")
        df_model[f"{col}_YEAR"] = dt.dt.year
        df_model[f"{col}_MONTH"] = dt.dt.month
        df_model[f"{col}_DAY"] = dt.dt.day
        df_model[f"{col}_DOW"] = dt.dt.dayofweek
        if col == "EXAM_START":
            df_model[f"{col}_HOUR"] = dt.dt.hour

# Keep missingness signal where available
for flag in ["RESULT_ID_MISSING", "SCORE_MISSING", "EXAM_START_MISSING", "CREDENTIAL_DATE_MISSING"]:
    if flag in df_model.columns:
        df_model[flag] = df_model[flag].astype("Int64")

# Drop raw datetime text columns after feature engineering
date_raw_cols = [c for c in ["EXAM_START", "ACCOUNT_SETUP_COMPLETE_DATE", "CREDENTIAL_DATE"] if c in df_model.columns]
df_model = df_model.drop(columns=date_raw_cols)

print("Dropped raw date columns:", date_raw_cols)
print("Shape after date engineering:", df_model.shape)


In [ ]:
# 3) Remove perfectly redundant missingness flags (keep only one representative)
redundant_flags = ["RESULT_ID_MISSING", "SCORE_MISSING", "EXAM_START_MISSING"]
existing_flags = [c for c in redundant_flags if c in df_model.columns]

if len(existing_flags) > 1:
    # Keep the first flag only
    keep_flag = existing_flags[0]
    drop_flags = existing_flags[1:]
    df_model = df_model.drop(columns=drop_flags)
    print("Kept missingness flag:", keep_flag)
    print("Dropped redundant flags:", drop_flags)
else:
    print("No redundant flags to drop")


In [ ]:
# 4) Split columns by type for preprocessing
numeric_cols = df_model.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_model.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))


In [ ]:
# 5) Build preprocessing pipeline
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols),
    ]
)

X = preprocessor.fit_transform(df_model)
print("Transformed matrix shape:", X.shape)


In [ ]:
# 6) Choose DBSCAN eps using k-distance plot
# Heuristic: min_samples around 2 * n_features_small can be large; use practical baseline first
min_samples = 20

nn = NearestNeighbors(n_neighbors=min_samples, metric="euclidean")
nn.fit(X)

distances, _ = nn.kneighbors(X)
k_dist = np.sort(distances[:, -1])

plt.figure(figsize=(10, 4))
plt.plot(k_dist)
plt.title(f"k-distance plot (k={min_samples})")
plt.xlabel("Sorted points")
plt.ylabel("Distance to k-th nearest neighbor")
plt.grid(alpha=0.3)
plt.show()

# Quick automatic starting guess (adjust after inspecting elbow)
eps_guess = float(np.quantile(k_dist, 0.95))
print("Suggested eps (95th percentile):", round(eps_guess, 4))


In [ ]:
# 7) Fit DBSCAN (tune eps/min_samples as needed)
eps = eps_guess

dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric="euclidean", n_jobs=-1)
labels = dbscan.fit_predict(X)

clustered = pd.Series(labels).value_counts().sort_index()
noise_ratio = float((labels == -1).mean())

print("Cluster label counts:")
print(clustered)
print("Noise ratio:", round(noise_ratio, 4))


In [ ]:
# 8) Attach cluster labels back to rows
result = df.copy()
result["DBSCAN_CLUSTER"] = labels

result[["DBSCAN_CLUSTER"]].value_counts().head(10)


In [ ]:
# 9) Optional 2D PCA projection for quick visual inspection
# (Only for visualization; clustering was done in full transformed feature space.)

sample_size = min(10000, X.shape[0])
rng = np.random.default_rng(42)
idx = rng.choice(np.arange(X.shape[0]), size=sample_size, replace=False)

X_sample = X[idx]
labels_sample = labels[idx]

pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_sample.toarray() if hasattr(X_sample, "toarray") else X_sample)

plt.figure(figsize=(8, 6))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels_sample, s=5, cmap="tab20", alpha=0.7)
plt.title("DBSCAN clusters (PCA projection)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(alpha=0.2)
plt.show()


## 10) DBSCAN tuning loop (eps/min_samples)
This section performs a lightweight parameter sweep and ranks configurations.


In [ ]:
# Tune DBSCAN across candidate eps and min_samples values
# Note: silhouette is computed only on non-noise points when >= 2 clusters exist.

min_samples_grid = [10, 15, 20, 25, 30]
eps_grid = np.quantile(k_dist, [0.85, 0.90, 0.93, 0.95, 0.97, 0.99])

tuning_rows = []

for ms in min_samples_grid:
    for eps_val in eps_grid:
        model = DBSCAN(eps=float(eps_val), min_samples=int(ms), metric="euclidean", n_jobs=-1)
        y = model.fit_predict(X)

        n_noise = int((y == -1).sum())
        noise_ratio = float((y == -1).mean())

        # Number of clusters excluding noise label -1
        cluster_labels = set(y)
        n_clusters = len(cluster_labels - {-1})

        sil = np.nan
        if n_clusters >= 2:
            mask = y != -1
            y_non_noise = y[mask]
            # Need at least 2 classes and some minimum records
            if len(np.unique(y_non_noise)) >= 2 and mask.sum() >= 100:
                X_eval = X[mask]
                # For large data, evaluate on a sample for speed
                max_eval = 8000
                if X_eval.shape[0] > max_eval:
                    rng = np.random.default_rng(42)
                    idx = rng.choice(np.arange(X_eval.shape[0]), size=max_eval, replace=False)
                    X_eval = X_eval[idx]
                    y_eval = y_non_noise[idx]
                else:
                    y_eval = y_non_noise

                try:
                    sil = silhouette_score(X_eval, y_eval, metric="euclidean")
                except Exception:
                    sil = np.nan

        tuning_rows.append({
            "eps": float(eps_val),
            "min_samples": int(ms),
            "n_clusters": int(n_clusters),
            "noise_ratio": noise_ratio,
            "noise_count": n_noise,
            "silhouette_non_noise": sil,
        })

tuning_df = pd.DataFrame(tuning_rows)

# Prefer runs with >=2 clusters and manageable noise, then best silhouette
ranked = tuning_df.copy()
ranked["silhouette_non_noise"] = ranked["silhouette_non_noise"].fillna(-999)
ranked = ranked.sort_values(
    by=["silhouette_non_noise", "n_clusters", "noise_ratio"],
    ascending=[False, False, True]
).reset_index(drop=True)

print("Top tuning candidates:")
ranked.head(10)


In [ ]:
# Pick best candidate from tuning table and re-fit final DBSCAN
best = ranked.iloc[0]
best_eps = float(best["eps"])
best_min_samples = int(best["min_samples"])

print("Selected parameters from tuning:")
print({"eps": best_eps, "min_samples": best_min_samples})

dbscan_tuned = DBSCAN(
    eps=best_eps,
    min_samples=best_min_samples,
    metric="euclidean",
    n_jobs=-1,
)
labels_tuned = dbscan_tuned.fit_predict(X)

print("Tuned label counts:")
print(pd.Series(labels_tuned).value_counts().sort_index())
print("Tuned noise ratio:", round(float((labels_tuned == -1).mean()), 4))


In [ ]:
# Save tuned labels to dataframe (optional)
result_tuned = df.copy()
result_tuned["DBSCAN_CLUSTER_TUNED"] = labels_tuned

result_tuned[["DBSCAN_CLUSTER_TUNED"]].value_counts().head(10)


## Notes for tuning
- If **noise ratio is too high** (e.g., > 60%), increase `eps` gradually.
- If **everything collapses into one cluster**, reduce `eps`.
- Try multiple `min_samples` values (e.g., 10, 20, 30) and compare stability.
- Consider dropping very sparse features and very high-cardinality categories if clustering is unstable.
